# Appendix: verbalized-confidence prompt selection (Table 3, `tab:vc_prompt_comp`)

The ten candidate prompt stems (`basc.verbal_confidence.VERBAL_CONF_STEMS`) are
compared on 4 models x {GSM8K, TriviaQA} (7 combinations: DeepSeek-R1-Distill-Llama-8B
is not evaluated on QA), over 10 random 40/60 splits
(metrics on the 60% test part). Unparseable outputs are imputed with the mean
confidence ("Imputed" = fraction imputed). Prompts are ranked on every metric;
the best average rank (stem 6) is used in all experiments.

Inputs: `outputs/.../*_verbal_conf_stem_{0..9}.csv` (`scripts/baselines/verbalized_confidence.py`).

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

from basc import paths, roster
from basc.metrics import correctness_metrics
from basc.verbal_confidence import load_verbal_conf_scores
from paper_utils import TABLES_DIR

MODELS = ["Qwen/Qwen3-1.7B", "Qwen/Qwen3-8B", "nvidia/Nemotron-Cascade-8B-Thinking",
          "deepseek-ai/DeepSeek-R1-Distill-Llama-8B"]
DATASETS = ["gsm8k", "trivia_qa"]
N_STEMS, N_TRIALS = 10, 10

In [ ]:
rows = []
for dataset in DATASETS:
    for model in [m for m in MODELS if dataset in roster.datasets_for(m)]:
        out_df = pd.read_csv(paths.test_responses(model, dataset, n=paths.N, pool_temp=paths.POOL_TEMP,
                                                  pool_k=paths.POOL_K))
        valid = out_df["answer"].notna() & (out_df["answer"].astype(str).str.strip() != "")
        out_df = out_df.loc[valid].reset_index(drop=True)
        for seed in range(N_TRIALS):
            _, test_df = train_test_split(out_df, test_size=0.6, random_state=seed, shuffle=True)
            correct = test_df["correct"].to_numpy(dtype=int)
            for stem in range(N_STEMS):
                vc = load_verbal_conf_scores(paths.verbal_confidence(model, dataset, stem_idx=stem),
                                             test_df["id"])
                if vc is None:
                    continue
                mask = np.isfinite(vc)
                vc = np.clip(np.where(mask, vc, float(vc[mask].mean()) if mask.any() else 0.5), 0.0, 1.0)
                rows.append({"Prompt": stem, **correctness_metrics(vc, correct),
                             "Imputed": float((~mask).sum() / len(vc)),
                             "Model": model, "Dataset": dataset, "seed": seed})
df = pd.DataFrame(rows)

In [ ]:
MINIMIZE = ["ECE1", "ECE2", "MCE", "Brier", "Imputed"]
MAXIMIZE = ["AUROC"]

per_prompt = df.groupby("Prompt")[MINIMIZE + MAXIMIZE].mean()
ranks = pd.concat([per_prompt[MINIMIZE].rank(ascending=True),
                   per_prompt[MAXIMIZE].rank(ascending=False)], axis=1)
per_prompt["Rank"] = ranks.mean(axis=1).rank(ascending=True)
table3 = per_prompt.sort_values("Rank").reset_index()[["Prompt"] + MINIMIZE + MAXIMIZE + ["Rank"]]

tex3 = table3.to_latex(index=False, float_format="%.3f")
(TABLES_DIR / "table3_vc_prompts.tex").write_text(tex3)
print(tex3)
table3